# 05 Robustness

Robustness check with a second buyer: PEPFAR/SCMS HIV shipments 2006-2015 (USAID).

Compares SCMS with the Global Fund PQR restricted to the same scope (HIV/ARV medicines,
purchase/delivery years 2006-2015):
  1. supplier concentration and overlap of leading firms;
  2. product relatedness: correlation of Hidalgo proximity (phi) on common product pairs,
     with a Mantel-type permutation test;
  3. community structure: Louvain modularity vs 200 degree-preserving rewirings;
  4. entry: does density predict a firm's entry into new ARV products (cutoffs 2009, 2012)?
Firm = manufacturing site mapped with the same firm rules; product = molecule combination
(normalized so SCMS "Lamivudine/Zidovudine" == PQR "Lamivudine+Zidovudine - FDC").


f10: product proximity in Global Fund PQR vs PEPFAR SCMS (HIV ARVs, 2006-2015).

In [ ]:
%run 00_utils.ipynb

import re
import numpy as np, pandas as pd, networkx as nx, statsmodels.formula.api as smf, matplotlib.pyplot as plt
from networkx.algorithms.community import louvain_communities, modularity

set_style()

## Comparison

In [ ]:
rng = np.random.default_rng(5205)
TAB = ROOT / "outputs" / "tables"
MIN_TX = 2

ALIAS = {"tenofovir disoproxil fumarate": "tenofovir", "tenofovir df": "tenofovir", "tdf": "tenofovir",
         "didanosine ec": "didanosine", "nelfinavir as mesilate": "nelfinavir"}
def prod_key(s):
    s = s.lower().replace(" - fdc", " ").replace(" fdc", " ").replace("co-blister", " ")
    s = re.sub(r"\(.*?\)", " ", s)
    parts = [re.sub(r"[^a-z ]", " ", p).strip() for p in re.split(r"[+/\[\]]", s)]
    parts = [ALIAS.get(re.sub(r"\s+", " ", p), re.sub(r"\s+", " ", p)) for p in parts if p.strip()]
    return " + ".join(sorted(set(parts)))

def firm_of(raw):
    m = match(raw)
    return m[0] if m and m[0] else re.split(r"[,(]", raw)[0].strip()

In [ ]:
# ---- SCMS
s = pd.read_csv(ROOT / "data" / "raw" / "scms_delivery_history_2006-2015.csv")
s = s[s["product group"] == "ARV"].copy()
s["firm"] = s["manufacturing site"].map(firm_of)
s["product"] = s["molecule/test type"].map(prod_key)
s["year"] = pd.to_datetime(s["delivered to client date"], format="%d-%b-%y", errors="coerce").dt.year
s["value"] = pd.to_numeric(s["line item value"], errors="coerce")
s = s.rename(columns={"id": "pk"})

I used the USAID SCMS Delivery History, which has 10,324 PEPFAR-funded shipment lines from 2006-2015, including 8,550 ARV lines. The official source is now offline, so I took the file from the Internet Archive copy of the official CSV. I compared it with PQR HIV transactions in the same years and defined products as molecule combinations in both sources.

In [ ]:
# ---- PQR, same scope
p = pd.read_csv(ROOT / "data" / "processed" / "pqr_medicines_clean.csv")
p = p[(p.area == "HIV/ARV") & p.year.between(2006, 2015)].copy()
p["product"] = p["product"].map(prod_key); p["value"] = p.cost_usd

def links(df, min_tx=MIN_TX):
    e = df.groupby(["firm", "product"]).agg(n_tx=("pk", "size"), first_year=("year", "min")).reset_index()
    return e[e.n_tx >= min_tx]

rows = []
def add(section, metric, pqr_val, scms_val, note=""):
    rows.append({"section": section, "metric": metric, "PQR (Global Fund)": pqr_val, "SCMS (PEPFAR)": scms_val, "note": note})

In [ ]:
# 1. scope and concentration
for name, d in [("pqr", p), ("scms", s)]:
    globals()[f"sh_{name}"] = d.groupby("firm").value.sum() / d.value.sum()
add("scope", "transactions (ARV, 2006-2015)", len(p), len(s))
add("scope", "firms", p.firm.nunique(), s.firm.nunique())
add("scope", "products (molecule combinations)", p["product"].nunique(), s["product"].nunique())
add("scope", "value, USD m", round(p.value.sum() / 1e6), round(s.value.sum() / 1e6))
add("concentration", "top-5 firm share of value", round(sh_pqr.nlargest(5).sum(), 3), round(sh_scms.nlargest(5).sum(), 3))
add("concentration", "top-10 firm share of value", round(sh_pqr.nlargest(10).sum(), 3), round(sh_scms.nlargest(10).sum(), 3))
t5p, t5s = set(sh_pqr.nlargest(5).index), set(sh_scms.nlargest(5).index)
add("concentration", "top-5 firms", "; ".join(sh_pqr.nlargest(5).index), "; ".join(sh_scms.nlargest(5).index),
    f"{len(t5p & t5s)} of 5 in common")
common_f = sh_pqr.index.intersection(sh_scms.index)
add("concentration", "Spearman corr. of firm shares (common firms)",
    round(sh_pqr[common_f].corr(sh_scms[common_f], method="spearman"), 2), "", f"{len(common_f)} common firms")
cp = set(p["product"]) & set(s["product"])
add("scope", "products in both sources", len(cp), len(cp))

The two buyers saw a similar supplier market. The top 5 firms held 85% of value in PQR and 91% in SCMS, and 4 of the top 5 firms were the same. Firm shares across the 18 common firms had a Spearman correlation of 0.77.

In [ ]:
# 2. product relatedness
ep, es = links(p), links(s)
def phi_matrix(e, prods):
    pe = project_products(e)
    M = pd.DataFrame(0.0, index=prods, columns=prods)
    for a, b, f in pe[["source", "target", "phi"]].itertuples(index=False):
        if a in M.index and b in M.index: M.loc[a, b] = M.loc[b, a] = f
    return M
prods = sorted(set(ep["product"]) & set(es["product"]))
Mp, Ms = phi_matrix(ep, prods), phi_matrix(es, prods)
iu = np.triu_indices(len(prods), 1)
xp, xs = Mp.values[iu], Ms.values[iu]
r_obs = np.corrcoef(xp, xs)[0, 1]
perm = []
for _ in range(2000):
    o = rng.permutation(len(prods)); perm.append(np.corrcoef(xp, Ms.values[np.ix_(o, o)][iu])[0, 1])
p_mantel = (np.sum(np.array(perm) >= r_obs) + 1) / (len(perm) + 1)
add("relatedness", "Pearson corr. of product proximity (common pairs)", round(r_obs, 2), "",
    f"{len(prods)} common products, {len(xp)} pairs; Mantel permutation p = {p_mantel:.4f}")
add("relatedness", "Spearman corr. of product proximity", round(pd.Series(xp).corr(pd.Series(xs), method="spearman"), 2), "")
pd.DataFrame({"pair": [f"{prods[i]} | {prods[j]}" for i, j in zip(*iu)], "phi_pqr": xp, "phi_scms": xs}).to_csv(
    ROOT / "data" / "processed" / "network" / "scms_vs_pqr_phi.csv", index=False)

Product relatedness also looked similar. Proximity across 406 common product pairs correlated at r = 0.78 (Mantel p < 0.001). This suggests relatedness reflects what firms can make, not only Global Fund procurement rules.

In [ ]:
# 3. community structure vs null
def graph(e):
    G = nx.Graph(); G.add_nodes_from(e["product"].unique())
    G.add_weighted_edges_from(project_products(e)[["source", "target", "phi"]].itertuples(index=False)); return G
def best_q(G, seeds):
    return max(modularity(G, c, weight="weight") for c in (louvain_communities(G, weight="weight", seed=int(x)) for x in seeds))
def rewire(e, n_swaps):
    f, q = e["firm"].to_numpy().copy(), e["product"].to_numpy().copy(); have = set(zip(f, q)); m = len(f)
    for _ in range(n_swaps):
        i, j = rng.integers(m, size=2)
        if f[i] == f[j] or q[i] == q[j] or (f[i], q[j]) in have or (f[j], q[i]) in have: continue
        have -= {(f[i], q[i]), (f[j], q[j])}; q[i], q[j] = q[j], q[i]; have |= {(f[i], q[i]), (f[j], q[j])}
    return pd.DataFrame({"firm": f, "product": q})
for name, e in [("PQR (Global Fund)", ep), ("SCMS (PEPFAR)", es)]:
    G = graph(e); q = best_q(G, range(30))
    null = [best_q(graph(rewire(e, 10 * len(e))), rng.integers(1e6, size=3)) for _ in range(200)]
    z = (q - np.mean(null)) / np.std(null)
    rows.append({"section": "communities", "metric": f"modularity Q / null mean / z  [{name}]",
                 "PQR (Global Fund)" if name.startswith("PQR") else "SCMS (PEPFAR)": f"{q:.3f} / {np.mean(null):.3f} / {z:.1f}",
                 "note": f"p = {(np.sum(np.array(null) >= q) + 1) / 201:.3f}; {G.number_of_nodes()} products, {e.firm.nunique()} firms"})

Within HIV alone, neither network had communities beyond chance (PQR Q = 0.12 vs null 0.13; SCMS Q = 0.07 vs null 0.11). This fits the main analysis, where HIV products form one dense core. The communities only appear when all three diseases are pooled.

In [ ]:
# 4. entry vs density (SCMS only)
def panel(lk, T, H=3, recent_df=None):
    st = lk[lk.first_year < T]; pe = project_products(st)
    phi = pd.concat([pe.rename(columns={"source": "p", "target": "q"}), pe.rename(columns={"target": "p", "source": "q"})])
    tot = phi.groupby("p").phi.sum(); have = set(zip(st.firm, st["product"]))
    act = recent_df[recent_df.year.between(T - 3, T - 1)]
    g = pd.MultiIndex.from_product([act.firm.unique(), act["product"].unique()], names=["firm", "product"]).to_frame(index=False)
    g = g[[(f, q) not in have for f, q in zip(g.firm, g["product"])]]
    cov = st[["firm", "product"]].rename(columns={"product": "q"}).merge(phi, on="q").groupby(["firm", "p"]).phi.sum().rename("cov").reset_index()
    g = g.merge(cov, left_on=["firm", "product"], right_on=["firm", "p"], how="left").drop(columns="p")
    g["density"] = (g["cov"].fillna(0) / g["product"].map(tot)).fillna(0)
    g["firm_n"] = g.firm.map(st.groupby("firm")["product"].nunique()).fillna(0)
    g["prod_n"] = g["product"].map(st.groupby("product").firm.nunique()).fillna(0)
    fut = set(zip(*lk[(lk.first_year >= T) & (lk.first_year < T + H)][["firm", "product"]].T.values))
    g["entry"] = [int((f, q) in fut) for f, q in zip(g.firm, g["product"])]; g["cutoff"] = T
    return g
lk = s.groupby(["firm", "product"]).agg(first_year=("year", "min")).reset_index()
pan = pd.concat([panel(lk, T, recent_df=s) for T in [2009, 2012]])
pan = pan[(pan.firm_n > 0) & (pan.prod_n > 0)]
pan["density_z"] = (pan.density - pan.density.mean()) / pan.density.std()
pan["log_firm_n"], pan["log_prod_n"] = np.log(pan.firm_n), np.log(pan.prod_n)
m = smf.logit("entry ~ density_z + log_firm_n + log_prod_n + C(cutoff)", data=pan).fit(disp=0)
ci = np.exp(m.conf_int().loc["density_z"])
add("entry", "rows / entries (cutoffs 2009, 2012)", "", f"{len(pan)} / {pan.entry.sum()}")
add("entry", "median density: entrants vs non-entrants", "",
    f"{pan[pan.entry == 1].density.median():.2f} vs {pan[pan.entry == 0].density.median():.2f}")
add("entry", "OR per SD of density (logit, controls)", "1.37 [1.24, 1.52] (all diseases, t10 M4)",
    f"{np.exp(m.params['density_z']):.2f} [{ci[0]:.2f}, {ci[1]:.2f}], p = {m.pvalues['density_z']:.3f}")

out = pd.DataFrame(rows); out.to_csv(TAB / "t13_robustness_scms.csv", index=False)
pd.set_option("display.width", 250, "display.max_colwidth", 80)
print(out.fillna("").to_string(index=False))
print("\nSCMS unmatched firm names:", sorted(set(s.firm) - set(pd.read_csv(ROOT / "data/processed/firm_crosswalk.csv").firm.dropna())))

Entry followed density in PEPFAR purchases too (median density 0.39 for entrants vs 0.05 for non-entrants). One SD of density had an OR of 1.47 (95% CI 1.04-2.09), close to the 1.37 I found in PQR. There were only 39 entries, and the PQR number covers all diseases, so the comparison is rough. This check also only covers HIV up to 2015, with 18 firms and 29 products.

## Figure

In [ ]:
INK, INK2, MUTED, GRID, BLUE = "#0b0b0b", "#595959", "#8a8984", "#e6e5e1", "#003D78"
d = pd.read_csv(ROOT / "data" / "processed" / "network" / "scms_vs_pqr_phi.csv")
r = d.phi_pqr.corr(d.phi_scms)
fig, ax = plt.subplots(figsize=(5.8, 5.4))
ax.plot([0, 1], [0, 1], color=MUTED, lw=0.8, ls="--", zorder=1)
ax.scatter(d.phi_pqr, d.phi_scms, s=26, color=BLUE, alpha=0.55, edgecolors="white", linewidths=0.6, zorder=2)
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
ax.set_xlabel("Proximity in Global Fund purchases (PQR)"); ax.set_ylabel("Proximity in PEPFAR purchases (SCMS)")
ax.text(0.03, 0.95, f"r = {r:.2f}\n{len(d)} product pairs\nMantel p < 0.001", transform=ax.transAxes, va="top", fontsize=9, color=INK)
ax.set_title("Product relatedness looks the same to a different buyer", loc="left", fontsize=11.5, fontweight="bold", pad=22)
ax.text(0, 1.02, "Each dot = a pair of ARV products; proximity = how often the same firms supply both",
        transform=ax.transAxes, fontsize=8.5, color=INK2, va="bottom")
fig.text(0.01, -0.03, "HIV ARVs, 2006-2015, 29 molecule combinations bought by both. Dashed line = equal proximity.\n"
         "Sources: Global Fund PQR; USAID SCMS Delivery History (archived copy).", fontsize=7.5, color=MUTED)
fig.savefig(ROOT / "outputs" / "figures" / "f10_robustness_scms.png"); plt.show(); print("ok")